# Week 5 - Machine Learning: Regression & Classification
**NAVTAC AI/ML Course** | Tasks 52-65

| Day | Topics |
|-----|--------|
| Day 1 | Multivariate Linear Regression |
| Day 2 | Polynomial Regression, Overfitting |
| Day 3 | Logistic Regression |
| Day 4 | Decision Trees |
| Day 5 | K-Nearest Neighbors, Evaluation Metrics |

---
## Task 52 - Multivariate Linear Regression

In [ ]:
import numpy as np

print('=== Multivariate Linear Regression ===')
print('Model: h(x) = w0 + w1*x1 + w2*x2 + ... + wn*xn')
print('       or in matrix form: h = X @ W (including bias)')

# Dataset: House price prediction
# Features: [size_sqft, bedrooms, bathrooms, age_years]
np.random.seed(42)

X = np.array([
    [1500, 3, 2, 5],
    [2000, 4, 3, 8],
    [1200, 2, 1, 15],
    [1800, 3, 2, 3],
    [2500, 5, 4, 1],
    [1100, 2, 1, 20],
    [2200, 4, 3, 7],
    [1600, 3, 2, 10],
    [3000, 5, 4, 2],
    [1400, 2, 2, 12]
], dtype=float)

y = np.array([250, 350, 180, 290, 420, 150, 370, 260, 500, 210], dtype=float)

m, n = X.shape
print(f'Dataset: {m} samples, {n} features')
print('Features: size(sqft), bedrooms, bathrooms, age(years)')
print('Target: price (thousands PKR)')

# Feature scaling (z-score normalization)
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0)
X_norm = (X - X_mean) / X_std
print(f'\nOriginal X[0]: {X[0]}')
print(f'Normalized X[0]: {X_norm[0].round(4)}')

# Add bias term (column of ones)
X_b = np.hstack([np.ones((m, 1)), X_norm])

# Gradient Descent
W = np.zeros(n + 1)
alpha  = 0.1
epochs = 1000

for epoch in range(epochs):
    y_pred = X_b @ W
    cost   = (1/(2*m)) * np.sum((y_pred - y)**2)
    grad   = (1/m) * X_b.T @ (y_pred - y)
    W     -= alpha * grad
    if epoch % 200 == 0:
        print(f'  Epoch {epoch:4d}: cost={cost:.4f}')

print(f'\nTrained weights (with bias): {W.round(4)}')
y_pred = X_b @ W

mse  = np.mean((y_pred - y)**2)
rmse = mse**0.5
ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - np.mean(y))**2)
r2   = 1 - ss_res/ss_tot

print(f'\nMSE  = {mse:.2f}')
print(f'RMSE = {rmse:.2f}')
print(f'R2   = {r2:.4f}')

print('\nActual vs Predicted:')
for i in range(m):
    print(f'  actual={y[i]:.0f}  predicted={y_pred[i]:.1f}')

---
## Task 53 - Normal Equation & sklearn Comparison

In [ ]:
import numpy as np

print('=== Normal Equation (Analytical Solution) ===')
print('W = (X^T X)^-1 X^T y')

X = np.array([
    [1500, 3, 2, 5],
    [2000, 4, 3, 8],
    [1200, 2, 1, 15],
    [1800, 3, 2, 3],
    [2500, 5, 4, 1],
    [1100, 2, 1, 20],
    [2200, 4, 3, 7],
    [1600, 3, 2, 10],
    [3000, 5, 4, 2],
    [1400, 2, 2, 12]
], dtype=float)
y = np.array([250, 350, 180, 290, 420, 150, 370, 260, 500, 210], dtype=float)

X_b = np.c_[np.ones(len(X)), X]
W = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
print(f'W (bias, w1..w4): {W.round(4)}')

y_pred = X_b @ W
r2 = 1 - np.sum((y-y_pred)**2)/np.sum((y-np.mean(y))**2)
print(f'R2 = {r2:.4f}')

# New prediction
new_house = np.array([[1, 1750, 3, 2, 6]])
pred = new_house @ W
print(f'\nPrediction for [size=1750, beds=3, baths=2, age=6]: {pred[0]:.2f}K')

print('\n=== Sklearn Implementation ===')
try:
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred_sk = model.predict(X_test)
    mse_sk = mean_squared_error(y_test, y_pred_sk)
    r2_sk  = r2_score(y_test, y_pred_sk)

    print(f'Intercept : {model.intercept_:.4f}')
    print(f'Coeff     : {model.coef_.round(4)}')
    print(f'Test MSE  : {mse_sk:.2f}')
    print(f'Test R2   : {r2_sk:.4f}')

    print('\nTest Predictions vs Actual:')
    for a, p in zip(y_test, y_pred_sk):
        print(f'  actual={a:.0f}  predicted={p:.1f}')

except ImportError:
    print('sklearn not installed - pip install scikit-learn')

---
## Task 54 - Polynomial Regression & Overfitting

In [ ]:
import numpy as np

print('=== Polynomial Regression ===')

# Non-linear dataset
np.random.seed(42)
X = np.linspace(-3, 3, 30)
y = 2*X**3 - 3*X**2 + X + 5 + np.random.randn(30) * 2

def poly_features(X, degree):
    return np.column_stack([X**d for d in range(degree+1)])

print('Degree | MSE')
print('-' * 30)
for deg in [1, 2, 3, 4, 7]:
    X_poly = poly_features(X, deg)
    W = np.linalg.pinv(X_poly.T @ X_poly) @ X_poly.T @ y
    y_pred = X_poly @ W
    mse = np.mean((y - y_pred)**2)
    r2  = 1 - np.sum((y-y_pred)**2)/np.sum((y-np.mean(y))**2)
    overfitting = ' (overfit risk)' if deg > 4 else ''
    print(f'  deg={deg}: MSE={mse:.4f}  R2={r2:.4f}{overfitting}')

print()
try:
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_squared_error, r2_score

    X_r = X.reshape(-1, 1)
    X_train, X_test, y_train, y_test = train_test_split(X_r, y, test_size=0.2, random_state=42)

    print('Degree | Train MSE | Test MSE  | R2')
    print('-' * 55)
    for deg in [1, 2, 3, 5]:
        pipe = Pipeline([
            ('poly', PolynomialFeatures(degree=deg)),
            ('lr',   LinearRegression())
        ])
        pipe.fit(X_train, y_train)
        tr_mse = mean_squared_error(y_train, pipe.predict(X_train))
        te_mse = mean_squared_error(y_test,  pipe.predict(X_test))
        r2 = r2_score(y_test, pipe.predict(X_test))
        ovf = ' OVERFIT' if (te_mse > tr_mse * 5) else ''
        print(f'  deg={deg}:  {tr_mse:8.4f}   {te_mse:8.4f}   {r2:.4f}{ovf}')

except ImportError:
    print('sklearn not installed - pip install scikit-learn')

---
## Task 55 - Logistic Regression (Binary Classification)

In [ ]:
import numpy as np

print('=== Logistic Regression ===')
print('Sigmoid: sigma(z) = 1 / (1 + e^-z)')
print('Model  : h(x) = sigma(W^T x)')
print('Cost   : J = -(1/m)*sum[y*log(h)+(1-y)*log(1-h)]  [Binary Cross-Entropy]')

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Test sigmoid
z_vals = [-3, -1, 0, 1, 3]
print('\nSigmoid values:')
for z in z_vals:
    print(f'  sigma({z:+}) = {sigmoid(z):.4f}')

# Dataset: Pass/Fail based on study hours + sleep hours
np.random.seed(42)
X = np.array([
    [1, 8], [2, 7], [3, 6], [4, 7], [5, 8],
    [6, 7], [7, 6], [8, 7], [9, 8], [10, 7],
    [1, 5], [2, 4], [3, 5], [4, 4], [5, 5],
    [6, 4], [7, 5], [8, 4], [9, 5], [10, 4]
], dtype=float)
y = np.array([0,0,0,0,0,1,1,1,1,1, 0,0,0,0,0, 1,1,1,1,1], dtype=float)

m, n = X.shape

# Feature normalization
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
X_b    = np.c_[np.ones(m), X_norm]

# Initialize weights
W = np.zeros(n + 1)
alpha  = 0.5
epochs = 1000

for epoch in range(epochs):
    z      = X_b @ W
    h      = sigmoid(z)
    cost   = -(1/m) * np.sum(y*np.log(h+1e-9) + (1-y)*np.log(1-h+1e-9))
    grad   = (1/m) * X_b.T @ (h - y)
    W     -= alpha * grad
    if epoch % 200 == 0:
        print(f'  Epoch {epoch:4d}: cost={cost:.4f}')

print(f'\nFinal weights: {W.round(4)}')

# Predictions
y_pred_prob = sigmoid(X_b @ W)
y_pred      = (y_pred_prob >= 0.5).astype(int)

accuracy = np.mean(y_pred == y)
tp = np.sum((y_pred == 1) & (y == 1))
fp = np.sum((y_pred == 1) & (y == 0))
tn = np.sum((y_pred == 0) & (y == 0))
fn = np.sum((y_pred == 0) & (y == 1))

print(f'\nAccuracy : {accuracy:.2%}')
print(f'Confusion Matrix:')
print(f'  TP={tp}  FP={fp}')
print(f'  FN={fn}  TN={tn}')
precision = tp/(tp+fp) if (tp+fp) > 0 else 0
recall    = tp/(tp+fn) if (tp+fn) > 0 else 0
f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1 Score : {f1:.4f}')

---
## Task 56 - Logistic Regression with sklearn (Iris Dataset)

In [ ]:
try:
    from sklearn.datasets import load_iris, load_breast_cancer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import (accuracy_score, classification_report,
                                  confusion_matrix, roc_auc_score)
    import numpy as np

    print('=== Binary Classification - Breast Cancer ===')
    bc = load_breast_cancer()
    X_bc, y_bc = bc.data, bc.target
    print(f'Shape: {X_bc.shape}  Classes: {bc.target_names}')

    X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    y_pr = lr.predict(X_te)

    print(f'Accuracy : {accuracy_score(y_te, y_pr):.4f}')
    print('Confusion Matrix:\n', confusion_matrix(y_te, y_pr))
    print('\nClassification Report:')
    print(classification_report(y_te, y_pr, target_names=bc.target_names))

    cv_scores = cross_val_score(lr, X_bc, y_bc, cv=5, scoring='accuracy')
    print(f'Cross-Val (5-fold): {cv_scores.round(4)}')
    print(f'Mean CV Acc: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

    print('\n=== Multiclass - Iris ===')
    iris = load_iris()
    X_ir, y_ir = iris.data, iris.target
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_ir, y_ir, test_size=0.2, random_state=42)
    sc2 = StandardScaler()
    X_tr2 = sc2.fit_transform(X_tr2)
    X_te2 = sc2.transform(X_te2)

    lr2 = LogisticRegression(max_iter=200, random_state=42)
    lr2.fit(X_tr2, y_tr2)
    y_pr2 = lr2.predict(X_te2)

    print(f'Accuracy (Iris 3-class): {accuracy_score(y_te2, y_pr2):.4f}')
    print('Classification Report:')
    print(classification_report(y_te2, y_pr2, target_names=iris.target_names))

except ImportError:
    print('sklearn not installed - pip install scikit-learn')

---
## Task 57 - Decision Trees

In [ ]:
import numpy as np

print('=== Decision Tree Concepts ===')
print('Key Terms:')
print('  - Node        : test on a feature')
print('  - Branch      : outcome of test')
print('  - Leaf        : final prediction')
print('  - Depth       : length of longest path from root to leaf')
print()
print('Splitting Criteria:')
print('  - Gini Impurity = 1 - sum(p_i^2)')
print('  - Entropy = -sum(p_i * log2(p_i))')
print('  - Information Gain = Entropy(parent) - weighted Entropy(children)')

# Manual Gini and Entropy
def gini(probs):
    return 1 - sum(p**2 for p in probs)

def entropy(probs):
    import math
    return -sum(p * math.log2(p) for p in probs if p > 0)

print('\nGini Examples:')
for probs in [(1.0, 0.0), (0.5, 0.5), (0.7, 0.3)]:
    print(f'  p={probs} -> gini={gini(probs):.4f}  entropy={entropy(probs):.4f}')

try:
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.datasets import load_iris
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, classification_report

    iris = load_iris()
    X, y = iris.data, iris.target
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

    print('\n=== Decision Tree - Iris ===')
    for depth in [None, 2, 3, 5]:
        dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
        dt.fit(X_tr, y_tr)
        tr_acc = accuracy_score(y_tr, dt.predict(X_tr))
        te_acc = accuracy_score(y_te, dt.predict(X_te))
        depth_label = str(depth) if depth else 'none'
        print(f'  max_depth={depth_label:4}: train={tr_acc:.4f}  test={te_acc:.4f}')

    # Best model
    best = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
    best.fit(X_tr, y_tr)
    y_pr = best.predict(X_te)
    print('\nBest model (depth=3) report:')
    print(classification_report(y_te, y_pr, target_names=iris.target_names))

    # Feature importances
    print('Feature Importances:')
    for name, imp in zip(iris.feature_names, best.feature_importances_):
        print(f'  {name:<22}: {imp:.4f}  {"*"*int(imp*20)}')

except ImportError:
    print('sklearn not installed - pip install scikit-learn')

---
## Task 58 - K-Nearest Neighbors (KNN)

In [ ]:
import numpy as np

print('=== KNN Concepts ===')
print('1. Store all training examples')
print('2. For each new point:')
print('   a. Compute distance to all training points')
print('   b. Select K nearest neighbors')
print('   c. Classification: majority vote')
print('      Regression: average of K neighbors')
print()
print('Distance Metrics:')
print('  - Euclidean: sqrt(sum((xi-yi)^2))')
print('  - Manhattan: sum(|xi-yi|)')
print('  - Minkowski: generalization of both')

# Manual KNN
def euclidean_distance(a, b):
    return np.sqrt(np.sum((np.array(a) - np.array(b))**2))

def manhattan_distance(a, b):
    return np.sum(np.abs(np.array(a) - np.array(b)))

def knn_predict(X_train, y_train, x_new, k=3):
    distances = [(euclidean_distance(x_new, x_tr), y_train[i])
                 for i, x_tr in enumerate(X_train)]
    distances.sort(key=lambda t: t[0])
    k_neighbors = [label for _, label in distances[:k]]
    # Majority vote
    from collections import Counter
    return Counter(k_neighbors).most_common(1)[0][0]

# Simple 2D dataset
X_train = np.array([[1,2],[2,3],[3,1],[4,5],[5,4],[6,5],[7,6],[8,7]])
y_train = np.array([0,0,0,1,1,1,1,1])

test_points = [[3.5, 3.5], [6, 4], [2, 1]]
print('Manual KNN Predictions (k=3):')
for pt in test_points:
    pred = knn_predict(X_train, y_train, pt, k=3)
    print(f'  Point {pt} -> Class {pred}')

try:
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.datasets import load_iris
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import accuracy_score, classification_report

    iris = load_iris()
    X, y = iris.data, iris.target
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    print('\n=== KNN - Iris (sklearn) ===')
    print('K  | Accuracy')
    print('-' * 20)
    best_k, best_acc = 1, 0
    for k in [1, 3, 5, 7, 9, 11, 15]:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, knn.predict(X_te))
        flag = ' <- best' if acc > best_acc else ''
        if acc > best_acc: best_k, best_acc = k, acc
        print(f'  k={k:2d}:  {acc:.4f}{flag}')

    best = KNeighborsClassifier(n_neighbors=best_k)
    best.fit(X_tr, y_tr)
    y_pr = best.predict(X_te)
    print(f'\nBest k={best_k} Report:')
    print(classification_report(y_te, y_pr, target_names=iris.target_names))

except ImportError:
    print('sklearn not installed - pip install scikit-learn')

---
## Task 59 - Model Evaluation Metrics

In [ ]:
import numpy as np

print('=== Classification Metrics ===')

def confusion_matrix_manual(y_true, y_pred):
    tp = sum((yt==1) and (yp==1) for yt,yp in zip(y_true, y_pred))
    fp = sum((yt==0) and (yp==1) for yt,yp in zip(y_true, y_pred))
    tn = sum((yt==0) and (yp==0) for yt,yp in zip(y_true, y_pred))
    fn = sum((yt==1) and (yp==0) for yt,yp in zip(y_true, y_pred))
    return tp, fp, tn, fn

y_true = [1,1,1,1,1,0,0,0,0,0,1,0,1,0,1]
y_pred = [1,1,1,0,1,0,0,1,0,0,0,1,1,0,1]

tp, fp, tn, fn = confusion_matrix_manual(y_true, y_pred)

accuracy  = (tp+tn) / len(y_true)
precision = tp/(tp+fp) if (tp+fp) > 0 else 0
recall    = tp/(tp+fn) if (tp+fn) > 0 else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
specificity = tn/(tn+fp) if (tn+fp)>0 else 0

print(f'Confusion Matrix: TP={tp} FP={fp} TN={tn} FN={fn}')
print(f'Accuracy    : {accuracy:.4f}')
print(f'Precision   : {precision:.4f}')
print(f'Recall      : {recall:.4f}')
print(f'F1 Score    : {f1:.4f}')
print(f'Specificity : {specificity:.4f}')

print('\n=== Regression Metrics ===')

y_true_reg = np.array([3.5, 2.7, 4.1, 5.2, 3.8, 4.6, 2.9, 5.0, 3.2, 4.4])
y_pred_reg = np.array([3.2, 3.0, 4.3, 5.0, 3.5, 4.8, 3.1, 5.2, 3.0, 4.6])

errors = y_true_reg - y_pred_reg
mae    = np.mean(np.abs(errors))
mse    = np.mean(errors**2)
rmse   = mse**0.5
ss_res = np.sum(errors**2)
ss_tot = np.sum((y_true_reg - y_true_reg.mean())**2)
r2     = 1 - ss_res/ss_tot
n, p = len(y_true_reg), 1
adj_r2 = 1 - (1-r2)*(n-1)/(n-p-1)
mape   = np.mean(np.abs(errors/y_true_reg)) * 100

print(f'MAE        : {mae:.4f}')
print(f'MSE        : {mse:.4f}')
print(f'RMSE       : {rmse:.4f}')
print(f'R2         : {r2:.4f}')
print(f'Adj R2     : {adj_r2:.4f}')
print(f'MAPE       : {mape:.2f}%')

print('\n=== Cross-Validation ===')
try:
    from sklearn.model_selection import cross_val_score, KFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.datasets import load_iris
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline

    iris = load_iris()
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=200))])

    for cv_n in [3, 5, 10]:
        scores = cross_val_score(pipe, iris.data, iris.target, cv=cv_n, scoring='accuracy')
        print(f'  {cv_n}-fold CV: {scores.round(4)}  mean={scores.mean():.4f} std={scores.std():.4f}')

except ImportError:
    print('sklearn not available')

---
## Task 60 - Feature Engineering & Preprocessing

In [ ]:
import numpy as np

print('=== Feature Engineering ===')

data = {
    'age'     : [25, 30, 35, 28, 45, 22, 38, 55, 29, 42],
    'salary'  : [40000, 60000, 75000, 50000, 100000, 35000, 80000, 120000, 45000, 95000],
    'exp'     : [2, 5, 8, 3, 20, 1, 10, 30, 4, 18],
    'dept'    : ['IT','HR','IT','Finance','IT','HR','Finance','IT','HR','Finance'],
    'promoted': [0, 1, 1, 0, 1, 0, 1, 1, 0, 1]
}

import json
print('Dataset (first 3):')
for k, v in data.items():
    print(f'  {k:<12}: {v[:3]}...')

print('\n--- 1. Label Encoding ---')
depts = data['dept']
unique = sorted(set(depts))
label_map = {dept: i for i, dept in enumerate(unique)}
encoded = [label_map[d] for d in depts]
print(f'Label map: {label_map}')
print(f'Encoded  : {encoded}')

print('\n--- 2. One-Hot Encoding ---')
ohe = {}
for dept in unique:
    ohe[dept] = [1 if d==dept else 0 for d in depts]
for dept, vals in ohe.items():
    print(f'  {dept:<10}: {vals}')

print('\n--- 3. Feature Scaling ---')
age = np.array(data['age'], dtype=float)
sal = np.array(data['salary'], dtype=float)

# Min-Max
age_mm = (age - age.min()) / (age.max() - age.min())
# Z-score
sal_zs = (sal - sal.mean()) / sal.std()

print(f'age (raw)     : {age}')
print(f'age (min-max) : {age_mm.round(3)}')
print(f'sal (z-score) : {sal_zs.round(3)}')

print('\n--- 4. Feature Creation ---')
age_arr = np.array(data['age'])
sal_arr = np.array(data['salary'])
exp_arr = np.array(data['exp'])

sal_per_exp = sal_arr / (exp_arr + 1)
age_sq      = age_arr ** 2
is_senior   = (age_arr > 40).astype(int)

print(f'salary/exp      : {sal_per_exp.round(0)}')
print(f'age_squared     : {age_sq}')
print(f'is_senior(>40)  : {is_senior}')

try:
    from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
    print('\n--- sklearn Preprocessing ---')
    sc  = StandardScaler()
    mms = MinMaxScaler()
    le  = LabelEncoder()

    age_scaled = sc.fit_transform(age.reshape(-1,1)).flatten()
    print(f'StandardScaler age : {age_scaled.round(3)}')

    sal_mm = mms.fit_transform(sal.reshape(-1,1)).flatten()
    print(f'MinMaxScaler sal   : {sal_mm.round(3)}')

    dept_enc = le.fit_transform(depts)
    print(f'LabelEncoder dept  : {dept_enc}')

except ImportError:
    print('sklearn not installed')

---
## Tasks 61-65 - Full ML Pipeline (Titanic-like dataset)

In [ ]:
import numpy as np

print('=== Full ML Pipeline ===')

try:
    import pandas as pd
    from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report

    # Synthetic Titanic-like dataset
    np.random.seed(42)
    n = 200

    age    = np.random.randint(1, 80, n)
    pclass = np.random.choice([1,2,3], n, p=[0.2,0.3,0.5])
    sex    = np.random.choice([0,1], n)  # 0=female, 1=male
    fare   = np.random.exponential(30, n)

    # Survival: higher class + female + lower age -> more likely to survive
    logit  = 1.5*(pclass==1) + 0.8*(pclass==2) + 1.5*(sex==0) - 0.01*age + np.random.randn(n)*0.3
    survived = (logit > 0.5).astype(int)

    df = pd.DataFrame({'age':age,'pclass':pclass,'sex':sex,'fare':fare,'survived':survived})
    print(f'Dataset: {df.shape}')
    print(df.head())
    print(f'\nSurvival rate: {survived.mean():.2%}')

    X = df.drop('survived', axis=1).values
    y = df['survived'].values

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    models = {
        'Logistic Regression': LogisticRegression(max_iter=200),
        'Decision Tree'      : DecisionTreeClassifier(max_depth=5, random_state=42),
        'KNN'                : KNeighborsClassifier(n_neighbors=5),
        'Random Forest'      : RandomForestClassifier(n_estimators=50, random_state=42)
    }

    print('\nModel Comparison:')
    print(f'{"Model":<25} {"Train Acc":>10} {"Test Acc":>10} {"CV Mean":>10}')
    print('-' * 58)
    best_name, best_acc = '', 0
    for name, m in models.items():
        m.fit(X_tr_s, y_tr)
        tr_acc = accuracy_score(y_tr, m.predict(X_tr_s))
        te_acc = accuracy_score(y_te, m.predict(X_te_s))
        cv     = cross_val_score(m, X_tr_s, y_tr, cv=5).mean()
        flag = ' <-' if te_acc > best_acc else ''
        if te_acc > best_acc: best_name, best_acc = name, te_acc
        print(f'{name:<25} {tr_acc:>10.4f} {te_acc:>10.4f} {cv:>10.4f}{flag}')

    print(f'\nBest model: {best_name} (test acc={best_acc:.4f})')

    print('\n=== Hyperparameter Tuning with GridSearchCV ===')
    param_grid = {
        'n_estimators': [10, 50, 100],
        'max_depth'   : [3, 5, None],
        'min_samples_split': [2, 5]
    }
    rf = RandomForestClassifier(random_state=42)
    gs = GridSearchCV(rf, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
    gs.fit(X_tr_s, y_tr)
    print(f'Best params   : {gs.best_params_}')
    print(f'Best CV score : {gs.best_score_:.4f}')
    best_rf = gs.best_estimator_
    print(f'Test accuracy : {accuracy_score(y_te, best_rf.predict(X_te_s)):.4f}')

    print('\n=== Feature Importance (Random Forest) ===')
    features = ['age', 'pclass', 'sex', 'fare']
    for feat, imp in sorted(zip(features, best_rf.feature_importances_), key=lambda x: -x[1]):
        print(f'  {feat:<10}: {imp:.4f}  {"*"*int(imp*40)}')

except ImportError:
    print('Install: pip install scikit-learn pandas')

---
## Week 5 - All Tasks Complete!

| Task | Topic | Status |
|------|-------|--------|
| Task 52 | Multivariate Linear Regression (Gradient Descent) | DONE |
| Task 53 | Normal Equation + sklearn Linear Regression | DONE |
| Task 54 | Polynomial Regression & Overfitting | DONE |
| Task 55 | Logistic Regression (from scratch) | DONE |
| Task 56 | Logistic Regression (sklearn, Iris, Cancer) | DONE |
| Task 57 | Decision Trees (Gini, Entropy, feature importance) | DONE |
| Task 58 | K-Nearest Neighbors (manual + sklearn) | DONE |
| Task 59 | Evaluation Metrics (Confusion Matrix, F1, R2, CV) | DONE |
| Task 60 | Feature Engineering (scaling, encoding, creation) | DONE |
| Task 61-65 | Full ML Pipeline + GridSearchCV + Model Comparison | DONE |